Ensure the basic catalog structure is in place

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS league_pipeline;
CREATE SCHEMA IF NOT EXISTS league_pipeline.landing_zone;
CREATE SCHEMA IF NOT EXISTS league_pipeline.raw;
CREATE SCHEMA IF NOT EXISTS league_pipeline.bronze;
CREATE VOLUME IF NOT EXISTS league_pipeline.landing_zone.players;
CREATE VOLUME IF NOT EXISTS league_pipeline.landing_zone.checkpoints;

Auto Loader Job

In [0]:
from pyspark.sql.functions import col, current_timestamp

LANDING_PATH = "dbfs:/Volumes/league_pipeline/landing_zone/players/"
CHECKPOINT_SCHEMA = "dbfs:/Volumes/league_pipeline/landing_zone/checkpoints/players/schema"
CHECKPOINT_DATA = "dbfs:/Volumes/league_pipeline/landing_zone/checkpoints/players/data"
RAW_TABLE = "league_pipeline.raw.players"

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", CHECKPOINT_SCHEMA)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(LANDING_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_DATA)
    .trigger(availableNow=True)
    .toTable(RAW_TABLE)
)

query.awaitTermination()  # blocks until this batch fully finishes or throws

just_ingested = (
    spark.table(RAW_TABLE)
    .filter(col("_ingested_at") >= query.recentProgress[-1]["timestamp"]) # When the query started
    .select("_source_file")
    .distinct()
    .collect()
)

deleted, failed = [], []
for row in just_ingested:
    path = row["_source_file"]
    try:
        dbutils.fs.rm(path)
        deleted.append(path)
    except Exception as e:
        failed.append((path, str(e)))

print(f"Deleted {len(deleted)} files from landing zone.")
if failed:
    print(f"WARNING: {len(failed)} files failed to delete: {failed}")